In [1]:
import subprocess
import os
import geopandas as gpd
from pyproj import CRS
from tqdm import tqdm
import time
import fiona

# Connect to the Database

In [2]:
# Database connection (no username/password required)
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
pg_conn_string = f"PG:host={db_host} dbname={db_name} port={db_port}"

# User Input Required

In [4]:
#Input link to data to upload
input_path = r"N:\2265_Basildon Digital Evidence Base and Plan Making\Source IN\Data Download\poi_uk\poi_uk.gpkg"
# Input layer name on the geodatabase. Refer to naming guide saved here:
layer_name = "urb_cdrc_points_of_interest_poi_sep2024" 

#Input target projection (CRS). #Change to None if you dont know input projection
target_crs = CRS.from_epsg(27700)

#Define path the ogr2ogr.exe
ogr2ogr_path = r"C:\Program Files\QGIS 3.34.11\bin\ogr2ogr.exe"

# DO NOT CHANGE
gpkg_path = r"N:\Geodatabase\Raw_Data\GPKG_temp_folder_for_geodatabse_upload/temp_upload.gpkg"  # temp file for faster upload

# For Geojson, ERSI Shapefile or GPKG (with only 1 dataset)

In [5]:
# ---- STEP 1: Load and Inspect CRS ---- #
print("Reading input layer...")
gdf = gpd.read_file(input_path, engine="pyogrio")

if not gdf.crs:
    raise ValueError("Input file does not have a valid CRS defined.")

input_epsg = gdf.crs.to_epsg()
target_epsg = gpd.GeoSeries([gdf.geometry.iloc[0]], crs=target_crs).crs.to_epsg()

print(f"Detected input CRS: {input_epsg}")
print(f"Target CRS: {target_epsg}")

if input_epsg != target_epsg:
    print("Reprojecting to target CRS...")
    gdf = gdf.to_crs(target_crs)
else:
    print("Input CRS matches target CRS — skipping reprojection.")

# ---- STEP 2: Convert to GPKG if needed ---- #
ext = os.path.splitext(input_path)[1].lower()
if ext == ".gpkg":
    print("Input is already a GeoPackage — using original.")
    gpkg_path = input_path
else:
    print(f"Converting {ext} to GeoPackage for upload...")
    gdf.to_file(gpkg_path, driver="GPKG", layer=layer_name)

# ---- STEP 3: Geometry Validation ---- #
if not gdf.is_valid.all():
    print("Found invalid geometries — fixing with buffer(0)...")
    gdf["geometry"] = gdf.buffer(0)
    gdf.to_file(gpkg_path, driver="GPKG", layer=layer_name)

# ---- STEP 4: Upload to PostGIS ---- #
print("Uploading to PostGIS via ogr2ogr...")
start_time = time.time()

ogr2ogr_cmd = [
    ogr2ogr_path,
    "-f", "PostgreSQL",
    pg_conn_string,
    str(gpkg_path),                         # input file (GPKG)
    "-nln", f"{db_schema}.{layer_name}",   # name for the output table in PostGIS
    "-nlt", "PROMOTE_TO_MULTI",
    "-lco", "GEOMETRY_NAME=geometry",
    "-lco", "SPATIAL_INDEX=GIST",
    "-lco", "SCHEMA=" + db_schema,
    "-lco", "OVERWRITE=YES",
    "-overwrite"
]
with tqdm(total=100, desc="Uploading", bar_format="{l_bar}{bar}| {elapsed}") as pbar:
    result = subprocess.run(ogr2ogr_cmd, capture_output=True, text=True)
    pbar.update(100)

if result.returncode != 0:
    print("\n Error during ogr2ogr upload:")
    print(result.stderr)
    exit(1)
else:
    print("\n Upload completed successfully.")

print(f" Time taken: {time.time() - start_time:.2f} seconds")

Reading input layer...
Detected input CRS: 27700
Target CRS: 27700
Input CRS matches target CRS — skipping reprojection.
Input is already a GeoPackage — using original.
Uploading to PostGIS via ogr2ogr...


Uploading: 100%|██████████| 30:13


 Error during ogr2ogr upload:
ERROR 1: PROJ: proj_create_from_database: C:\Users\abhimanya.achara\Anaconda3\lib\site-packages\osgeo\data\proj\proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 1 whereas a number >= 3 is expected. It comes from another PROJ installation.
ERROR 1: In GetNextRawFeature(): sqlite3_step() : disk I/O error
ERROR 1: Terminating translation prematurely after failed
translation of layer poi_uk (use -skipfailures to skip errors)
ERROR 1: COPY statement failed.
server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

ERROR 1: no connection to the server

ERROR 1: no connection to the server

ERROR 1: no connection to the server

ERROR 1: no connection to the server

ERROR 1: no connection to the server

Warning 1: User lacks CREATE SCHEMA privilege to be able to create ogr_system_tables.metadata table
ERROR 1: no connection to the server

ERROR 1: sqlite3_get_table(SELECT md.metadata,

# For GPKG (with multiple dataset)

In [28]:
#Input link to data to upload
gpkg_path = r"N:\2265_Basildon Digital Evidence Base and Plan Making\Source IN\Data Download\Movement_Road Network\oproad_gb.gpkg"

# ---- Discover Layers in GPKG ---- #
print("Listing layers in GPKG...")
layers = fiona.listlayers(gpkg_path)
print(f"Found {len(layers)} layers.")
print(layers)

#Input target projection (CRS). #Change to None if you dont know input projection
target_crs = CRS.from_epsg(27700)

#Define path the ogr2ogr.exe
ogr2ogr_path = r"C:\Program Files\QGIS 3.34.11\bin\ogr2ogr.exe"

Listing layers in GPKG...
Found 3 layers.
['motorway_junction', 'road_link', 'road_node']


In [29]:
# ---- Step 1: Layers you want to upload ---- #
desired_layers = [
    #"building",
    #"important_building",
    #"named_place",
    #"railway_station",
    #"railway_track",
    #"railway_tunnel",
    #"surface_water_area",
    #"tidal_water"
    #'woodland'
    'road_link'
]

# ---- Step 2: Optional Postgres table names ---- #
layer_name_map = {
    #"building": "urb_os_buildings",
    #"important_building": "urb_os_important_buildings",
    #"named_place": "urb_os_named_places",
    #"railway_station": "mob_os_railway_stations",
    #"railway_track": "mob_os_railway_tracks",
    #"railway_tunnel": "mob_os_railway_tunnels",
    #"surface_water_area": "env_os_surface_water",
    #"tidal_water": "env_os_tidal_water"
    #"woodland": "env_os_woodland"
    'road_link': "mob_os_open_roads_apr2025"
}

In [30]:
# ---- Process Each Desired Layer ---- #
for layer_name in desired_layers:
    if layer_name not in layers:
        print(f"Layer '{layer_name}' not found in GPKG — skipping.")
        continue

    target_table_name = layer_name_map.get(layer_name, layer_name)
    print(f"\nProcessing layer: {layer_name} → {target_table_name}")

    try:
        # ---- Load and Check CRS ---- #
        gdf = gpd.read_file(gpkg_path, layer=layer_name)

        if not gdf.crs:
            raise ValueError(f"Layer '{layer_name}' does not have a valid CRS.")

        input_epsg = gdf.crs.to_epsg()
        target_epsg = gpd.GeoSeries([gdf.geometry.iloc[0]], crs=target_crs).crs.to_epsg()

        print(f"Input CRS: {input_epsg}, Target CRS: {target_epsg}")
        if input_epsg != target_epsg:
            print("Reprojecting to target CRS...")
            gdf = gdf.to_crs(target_crs)
        else:
            print("Input CRS matches target CRS — skipping reprojection.")

        # ---- Geometry Validation ---- #
        print(f"Geometry types: {gdf.geom_type.value_counts().to_dict()}")
        if not gdf.is_valid.all():
            print("Found invalid geometries — fixing with buffer(0)...")
            gdf["geometry"] = gdf.buffer(0)
            print("Validity after fix:")
            print(gdf.is_valid.value_counts())

        # ---- Save Cleaned GPKG Layer to Temp File ---- #
        temp_gpkg = fr"N:\Geodatabase\Raw_Data\GPKG_temp_folder_for_geodatabse_upload/{layer_name}.gpkg"
        gdf.to_file(temp_gpkg, layer=layer_name, driver="GPKG")

        # ---- Upload to PostGIS ---- #
        print(f"Uploading to PostGIS: {target_table_name}")
        start_time = time.time()

        ogr2ogr_cmd = [
            ogr2ogr_path,
            "-f", "PostgreSQL",
            pg_conn_string,
            temp_gpkg,
            layer_name,
            "-nln", f"{db_schema}.{target_table_name}",
            "-nlt", "PROMOTE_TO_MULTI",
            "-lco", "GEOMETRY_NAME=geometry",
            "-lco", "SPATIAL_INDEX=GIST",
            "-lco", "SCHEMA=" + db_schema,
            "-lco", "OVERWRITE=YES",
            "-overwrite"
        ]

        with tqdm(total=100, desc=f"Uploading {target_table_name}", bar_format="{l_bar}{bar}| {elapsed}") as pbar:
            result = subprocess.run(ogr2ogr_cmd, capture_output=True, text=True)
            pbar.update(100)

        if result.returncode != 0:
            print(f"Upload failed for {target_table_name}:\n{result.stderr}")
        else:
            print(f"Upload complete: {target_table_name} in {time.time() - start_time:.2f} seconds")

    except Exception as e:
        print(f"Failed to process '{layer_name}': {e}")


Processing layer: road_link → mob_os_open_roads_apr2025
Input CRS: 27700, Target CRS: 27700
Input CRS matches target CRS — skipping reprojection.
Geometry types: {'LineString': 3932885}


C:\Users\abhimanya.achara\Anaconda3\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


Uploading to PostGIS: mob_os_open_roads_apr2025


Uploading mob_os_open_roads_apr2025: 100%|██████████| 05:12

Upload complete: mob_os_open_roads_apr2025 in 312.82 seconds
